In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from src.chebae import chebae
from src.gaussian_search import qcels, qmegs
from src.compressed_sensing import compressed_sensing
from src.esprit import esprit_cosine, csae
from src.qpe import qpe
from src.util import CSAECache
import json

rng = np.random.default_rng()

In [ ]:
csae_cache = [CSAECache(i) for i in np.arange(1, 9)]
target = 1/3.0

In [ ]:
with open('./output/comparison/qpe.jsonl', 'w') as f:
    for j in range(100):
        res = [qpe(target, i, 20, rng) for i in 2 ** np.arange(1, 16)]
        y, x, z = zip(*res)
        json.dump({"query": np.array(x).tolist(),"value": np.array(y).tolist(), "depth": np.array(z).tolist()}, f)
        f.write("\n")

In [ ]:
with open('./output/comparison/chebae.jsonl', 'w') as f:
    for j in range(100):
        res = [chebae(target, 0.5 / i, 0.05, rng) for i in 2 ** np.arange(1, 16)]
        y, x, z = zip(*res)
        json.dump({"query": np.array(x).tolist(),"value": np.array(y).tolist(), "depth": np.array(z).tolist()}, f)
        f.write("\n")

In [ ]:
with open('./output/comparison/csae.jsonl', 'w') as f:
    for j in range(100):
        res = [csae(target, i, rng, 4, cache=csae_cache[i-1]) for i in np.arange(1, 8)]
        y, x, z = zip(*res)
        json.dump({"query": np.array(x).tolist(),"value": np.array(y).tolist(), "depth": np.array(z).tolist()}, f)
        f.write("\n")

In [ ]:
with open('./output/comparison/esprit.jsonl', 'w') as f:
    for j in range(100):
        res = [esprit_cosine(target, i, 20, rng) for i in 2 ** np.arange(1, 10)]
        y, x, z = zip(*res)
        json.dump({"query": np.array(x).tolist(),"value": np.array(y).tolist(), "depth": np.array(z).tolist()}, f)
        f.write("\n")

In [ ]:
with open('./output/comparison/sensing.jsonl', 'w') as f:
    for j in range(100):
        res = [compressed_sensing(target, i, i * np.max((2 * np.log2(i), 20)).astype(int), rng) for i in 2 ** np.arange(1, 16)]
        y, x, z = zip(*res)
        json.dump({"query": np.array(x).tolist(),"value": np.array(y).tolist(), "depth": np.array(z).tolist()}, f)
        f.write("\n")

In [ ]:
with open('./output/comparison/qcels.jsonl', 'w') as f:
    for j in range(100):
        res = [qcels(target, i, i * 20, rng) for i in 2 ** np.arange(0, 16)]
        y, x, z = zip(*res)
        json.dump({"query": np.array(x).tolist(),"value": np.array(y).tolist(), "depth": np.array(z).tolist()}, f)
        f.write("\n")

In [ ]:
with open('./output/comparison/qmegs.jsonl', 'w') as f:
    for j in range(100):
        res = [qmegs(target, i, i * 20, rng) for i in 2 ** np.arange(0, 16)]
        y, x, z = zip(*res)
        json.dump({"query": np.array(x).tolist(),"value": np.array(y).tolist(), "depth": np.array(z).tolist()}, f)
        f.write("\n")

In [ ]:
plt.rcParams.update({'font.size': 14})
plt.rcParams.update({'font.family': "serif"})
algos = ["QPE", "ChebAE", "CSAE", "ESPRIT", "Sensing", "QCELS", "QMEGS"]

In [ ]:
for algo in algos:
    with open(f'./output/comparison/{algo.lower()}.jsonl', 'r') as json_file:
        json_list = list(json_file)
    time = []
    value = []
    for json_str in json_list:
        res = json.loads(json_str)
        time.append(res["query"])
        value.append(res["value"])
    time = np.max(time,axis=0)
    error = np.median(np.abs(np.array(value)-target), axis=0)
    plt.plot(time,error,label=algo)
plt.xscale('log')
plt.yscale('log')
plt.xlabel("Queries")
plt.ylabel("Error")
plt.legend()
plt.tight_layout()
plt.savefig("compare_query.pdf")

In [ ]:
for algo in algos:
    with open(f'./output/comparison/{algo.lower()}.jsonl', 'r') as json_file:
        json_list = list(json_file)
    time = []
    value = []
    for json_str in json_list:
        res = json.loads(json_str)
        time.append(res["depth"])
        value.append(res["value"])
    time = np.max(time,axis=0)
    error = np.median(np.abs(np.array(value)-target), axis=0)
    plt.plot(time,error,label=algo)
plt.xscale('log')
plt.yscale('log')
plt.xlabel("Depth")
plt.ylabel("Error")
plt.legend()
plt.tight_layout()
plt.savefig("compare_depth.pdf")